# 260311 LLM 서비스를 위한 파이썬 기본기 총정리

**오늘의 목표**
- 어제 본 OpenAI 호출(`messages=[{role, content}, ...]`)을 **자유롭게 조립**할 수 있도록 파이썬 기초를 한 번에 훑는다.
- 변수/타입 → f-string → 리스트/for/if → 딕셔너리 → 함수 → try-except → 파일 I/O(JSON)까지.
- 모든 문법을 "왜 LLM 서비스에 필요한가?" 관점에서 본다. 사실상 LLM 서비스 = 문자열과 딕셔너리를 잘 다루는 일이다.

**비유 한 줄**: 파이썬 문법은 요리 도구다. 어제는 오븐(OpenAI API)을 켜봤고, 오늘은 도마·칼·계량컵 사용법을 익혀 재료(프롬프트)를 원하는 모양으로 손질한다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w1_api_and_chain/llm_260311_python_tutorial.ipynb)

## 0. 환경 세팅

오늘은 순수 파이썬 문법 위주라 API 호출은 거의 없지만, 마지막에 messages 리스트를 만들어볼 때 쓸 수 있도록 어제처럼 환경을 잡아둔다.

In [ ]:
# 필요 라이브러리 설치 (이미 설치되어 있으면 아무 일도 안 함)
!pip install -q openai langchain-openai langchain-core python-dotenv

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

## 1. 변수와 타입 — 정보의 생김새 알아보기

파이썬 변수는 타입을 **선언하지 않고** 그냥 대입하면 된다. 대신 `type()`으로 언제든 실제 타입을 확인할 수 있다.

- `str` (문자열) — 프롬프트, 답변 내용
- `int` (정수) — `max_tokens`, 인덱스
- `float` (실수) — `temperature`, `top_p`
- `bool` (참/거짓) — `stream` 같은 스위치

**비유 한 줄**: 변수는 택배 상자의 라벨이다. 내용물(값)이 뭔지에 따라 `str/int/float/bool` 같은 '품목 카테고리'가 정해진다.

In [ ]:
# 기본 네 가지 타입 한 번에 — 변수에 '='로 값을 꽂으면 끝
name = '김민아'        # str  (문자열)
age = 30               # int  (정수)
height = 168.8         # float(실수)
isStudent = False      # bool (True/False)

print("이름 :", name)
print(type(name), type(age), type(height), type(isStudent))

In [ ]:
# LLM 호출 파라미터를 변수로 미리 저장해두는 연습 — 나중에 API 호출에 그대로 꽂는다
model = 'gpt-4o-mini'      # str — 모델명
temperature = 0.7          # float — 창의성(0은 결정적, 1에 가까울수록 창의적)
max_tokens = 100           # int — 출력 토큰 상한 (비용 제어의 핵심!)
stream = True              # bool — 한 글자씩 흘려받을지 여부
top_p = 0.9                # float — 상위 확률 누적 샘플링 (temperature 대체용)
frequency_penalty = 0.0    # float — 반복 단어 패널티

print(model, temperature, max_tokens, stream, top_p, frequency_penalty)

In [ ]:
# 각 변수의 타입 확인 — type()은 디버깅할 때 가장 자주 쓰는 함수 중 하나
print(type(model), type(temperature), type(max_tokens))
print(type(stream), type(top_p), type(frequency_penalty))

## 2. f-string — "LLM 서비스는 문자열 조립력"

강사님 한 마디: **"LLM 서비스는 곧 문자열을 잘 다루는 능력."** 프롬프트는 결국 문자열이고, 사용자 입력을 끼워넣어 문자열을 조립해서 모델에 보내는 게 일의 대부분이다.

f-string은 `f"...{변수}..."` 형식으로 중괄호 안에 파이썬 표현식을 바로 꽂을 수 있다. 삼중 따옴표 `'''...'''`를 쓰면 여러 줄 문자열도 쉽게 만든다.

**비유 한 줄**: f-string은 빈칸 채우기 학습지다. 템플릿을 미리 준비하고 빈칸({변수})만 그때그때 채워 완성된 프롬프트를 찍어낸다.

In [ ]:
# 가장 기본 f-string — 변수 값이 그 자리에 '치환'된다
name = '김민아'
print(f'안녕하세요! {name}님')

In [ ]:
# 여러 줄 프롬프트는 삼중 따옴표로 — 페르소나 시스템 프롬프트에 많이 씀
prompt = '''
당신은 영어권 네이티브 영어 전문가입니다.
비영어권을 가르치는 10년차 선생님입니다.
제가 하는 질문에 친절하게 영어로 답하며 영어를 가르쳐 줍니다.
'''
print(prompt)

In [ ]:
# 여러 변수를 한 템플릿에 채워넣기 — 사용자마다 다른 인사말/주제를 동적으로 조립
user_name = '김민아'
topic = 'python'
print(f'안녕하세요. {user_name}님')
print(f'오늘은 {topic}에 대해 알아보겠습니다.')

### 자주 쓰는 문자열 메서드 4인방
LLM 입력을 정제할 때 단골로 쓰는 메서드들이다.

- `strip()` — 앞뒤 공백 제거 (사용자 입력에 섞여들어온 공백/개행 치우기)
- `split()` — 구분자로 쪼개기 (CSV, 단계별 응답 파싱)
- `replace()` — 치환 (민감 정보 마스킹, 특수문자 제거)
- `len()` — 길이 (토큰 수 추정용 글자 수 세기)

In [ ]:
# 사용자 입력은 공백/개행이 지저분한 경우가 많아서 strip()이 거의 필수
category = '계정'
question = '    비밀번호를 잊어버렸습니다.   '   # 앞뒤에 공백이 잔뜩
prompt = f'''
카테고리: {category}
질문: {question.strip()}
답변형식: 단계별로 설명해 주세요.
'''
print(prompt)

In [ ]:
# split()으로 쪼개기 — '단계별 응답'을 단계 단위로 분리하는 데 자주 씀
response = '1단계: 설정 열기, 2단계: 비밀번호 변경, 3단계: 저장하기'
steps = response.split(', ')    # ', '를 기준으로 자르면 리스트로 반환
print(steps)
print(f'총 {len(steps)}단계')

## 3. 리스트 — 순서 있는 여러 개의 값

리스트는 대괄호 `[]`로 묶어 값을 순서대로 보관한다. 인덱스는 **0부터**. 음수 인덱스(`-1`)는 뒤에서부터 센다.

**LLM 서비스에서 어디 쓰나?**
- `messages = [{"role": ..., "content": ...}, ...]` — 대화 이력
- 여러 모델 후보 목록, 여러 질문 일괄 처리 등

**비유 한 줄**: 리스트는 번호표가 붙은 서랍장. 0번 서랍부터 순서대로 꺼낼 수 있고, `-1`번 서랍은 '가장 마지막'을 의미한다.

In [ ]:
# 인기 LLM 모델들을 리스트에 담아보기
models = ['gpt-4o', 'gpt-4o-mini', 'claude-3']
scores = [0.95, 0.87, 0.72]

print(models[2])      # 인덱스 2 → 세 번째 원소 'claude-3'
print(models[-1])     # 음수 인덱스 → 뒤에서 첫 번째 (=맨 끝)
print(models[-2])     # 뒤에서 두 번째

In [ ]:
# append()로 뒤에 추가 — 대화 이력에 새 메시지 누적할 때 자주 씀
models.append('gemini-2.5-flash')
print(models)
print(f'모델 개수: {len(models)}')

## 4. for 반복문 — 리스트를 하나씩 훑기

`for 변수 in 리스트:` 형식. 리스트 원소를 하나씩 꺼내 변수에 담아 반복한다. 인덱스까지 같이 필요하면 `enumerate()`로 감싸면 `(i, item)` 튜플을 준다.

**비유 한 줄**: for 루프는 택배 상자 풀기다. 박스(리스트)에서 한 개씩 꺼내 같은 작업(포장 뜯기)을 반복한다.

In [ ]:
# 기본 for — models 리스트를 하나씩 출력
for item in models:
    print(item)

In [ ]:
# enumerate로 인덱스까지 받기 — '몇 번째 메시지'가 필요할 때
for i, item in enumerate(models):
    print(i, item)

### for + if 조합 — 조건 필터링
문자열 `in`은 '포함 여부'를 알려준다. `'비밀번호' in question`은 question에 비밀번호란 단어가 들어있으면 True.

**실전 예**: FAQ 질문 리스트에서 특정 키워드가 들어간 것만 뽑기.

In [ ]:
# for + if 맛보기
words = ['abcde', 'abc', 'bcd', 'efg']
for item in words:
    if 'bc' in item:       # 'bc' 문자열이 item 안에 있으면
        print(item, '-> 포함')
    else:
        print(item, '-> 없음')

In [ ]:
# 실전: 비밀번호 관련 질문만 골라내기 (FAQ 분류 맛보기)
questions = [
    '비밀번호를 잊어버렸어요?',
    'gpt가 좋아요, claude가 좋아요?',
    '로그인이 안돼요?',
    '개발자님, 비밀번호를 찾아주세요?',
    '회사에서 어떤 일을 하나요?'
]

keyword = '비밀번호'
for i, question in enumerate(questions):
    if keyword in question:
        # 1-based 번호(i+1)로 보기 좋게 출력
        print(f'[{i+1}] {question}')

## 5. 딕셔너리 — key로 꺼내쓰는 창고

딕셔너리는 중괄호 `{}`로 쓰고, `key: value` 쌍으로 데이터를 저장한다. 리스트가 '번호 순서'라면 딕셔너리는 '라벨'로 꺼낸다.

**LLM 핵심**: OpenAI API의 **메시지 하나**가 바로 딕셔너리다 — `{"role": "user", "content": "..."}`.

**비유 한 줄**: 리스트가 번호순 신발장이라면, 딕셔너리는 이름표가 붙은 사물함이다.

In [ ]:
# 기본 딕셔너리 — key로 값을 꺼내기
person = {'name': '홍길동', 'age': 30}
print(person['name'])   # 'name' 키에 저장된 값
print(person['age'])    # 'age' 키에 저장된 값

In [ ]:
# OpenAI 메시지 하나는 정확히 이런 모양의 딕셔너리다
message = {
    'role': 'user',
    'content': '비밀번호를 잊어버렸어요'
}
print(message['role'], '|', message['content'])

# 기존 딕셔너리에 새 key를 추가하고 싶으면 그냥 대입
message['timestamp'] = '2026-03-11 00:00:00'
print(message)

In [ ]:
# 메시지 '여러 개'는 딕셔너리를 담은 리스트 — 이게 바로 어제 봤던 messages 구조
messages = [
    {'role': 'system',    'content': '당신은 IT 상담원입니다.'},
    {'role': 'user',      'content': '비밀번호를 잊어버렸어요.'},
    {'role': 'assistant', 'content': '비밀번호 재설정 방법을 안내해 드리겠습니다.'},
]

# 루프 돌면서 예쁘게 출력 — 마치 카카오톡 대화창 렌더링처럼
for msg in messages:
    print(f"[{msg['role']}] {msg['content']}")

### `dict[key]` vs `dict.get(key)` — 없을 때 죽느냐 참느냐
- `config['max_tokens']` — 키가 없으면 **KeyError**로 프로그램이 뻗음.
- `config.get('max_tokens', 100)` — 키가 없으면 **기본값(100)**을 돌려줌.

**비유 한 줄**: `[]`는 "반드시 있다고 약속한 서랍"을 여는 것, `.get()`은 "없으면 빈 손으로 돌아올게요"라는 여유 있는 접근.

In [ ]:
# 존재하는 key와 없는 key를 다르게 다뤄보기
config = {'model': 'gpt-4o-mini', 'temperature': 0.7}

print(config['model'])                          # OK — 값 반환
print(config.get('max_tokens'))                  # None — 없으면 그냥 None
print(config.get('max_tokens', 1000))            # 1000 — 기본값 지정 가능

## 6. 함수 — 반복되는 로직 한 번에 포장하기

`def 함수이름(매개변수):` 로 정의하고 `return`으로 값을 돌려준다. 인자에 기본값을 줄 수도 있다.

**왜 함수가 중요한가?**: LLM 서비스에서 "메시지 딕셔너리 만들기" / "프롬프트 조립" / "질문 여부 판별" 같은 짧은 로직을 함수로 뽑아두면, 나중에 수백 번 호출해도 한 곳만 고치면 된다.

**비유 한 줄**: 함수는 전자레인지의 '팝콘 버튼'이다. 온도·시간·회전을 매번 세팅하지 않고 버튼 하나로 재현 가능.

In [ ]:
# 가장 기본 함수 — 이름을 받아서 인사말 출력
def 인사(name):
    print(f"안녕하세요, {name} 님!")

인사('이상호')
인사('김민아')

In [ ]:
# 매개변수 여러 개 + return 값 — 두 번 인사 후 문자열을 돌려줌
def 인사여러번(name, num):
    print(f"안녕하세요, {name} 님!\n" * num)  # 문자열 * 정수는 '반복' 의미
    return f"안녕하세요, {name} 님! 이 문장을 {num}번 출력했습니다."

결과 = 인사여러번('김민아', 3)
print('[반환값]', 결과)

In [ ]:
# LLM용 유틸 함수 1: 메시지 딕셔너리 만들기 — 매번 수동으로 dict 쓰는 수고 줄이기
def make_message(role_value, content_value):
    return {'role': role_value, 'content': content_value}

msg = make_message('user', '비밀번호가 없습니다')
print(msg)

In [ ]:
# LLM용 유틸 함수 2: 기본값이 있는 매개변수 (language='한국어')
def make_prompt(question, language='한국어'):
    """주어진 질문을 지정 언어로 답하도록 감싸는 프롬프트 조립기."""
    return f"다음 질문에 {language}로 답변해주세요 : {question}"

print(make_prompt('비밀번호 재설정 방법'))                 # 기본값 한국어
print(make_prompt('Password reset steps', 'English'))    # 영어로 바꾸기

In [ ]:
# LLM용 유틸 함수 3: 질문인지 판별 — 물음표가 끝에 있거나 끝이 '~요'로 끝나면 질문
def is_question(text):
    # or 연산자는 둘 중 하나만 True여도 True
    return text.strip().endswith('?') or '요' in text[-3:]

print(is_question('안녕하세요, 롯데월드는 어떻게 가나요?'))   # True
print(is_question('안녕하세요, 롯데월드는 어떻게 가나요'))    # True ('요' 끝)
print(is_question('오늘 날씨가 좋다'))                        # False

## 7. try / except — 에러가 나도 프로그램이 죽지 않게

API 호출은 네트워크/타임아웃/키 오류 등으로 언제든 실패할 수 있다. `try`에 위험한 코드를 넣고 `except`에 "에러 나면 뭐 할지"를 정의한다.

**비유 한 줄**: try-except는 자동차의 에어백이다. 사고(에러)가 나는 건 막지 못해도, 운전자(프로그램)가 죽지 않게 보호한다.

In [ ]:
# 0으로 나누기 — 평소엔 ZeroDivisionError로 크래시
try:
    result = 10 / 0
except ZeroDivisionError:
    print('0으로 나눌 수 없습니다.')
    result = None  # 실패 시 안전한 기본값

print('result =', result)

In [ ]:
# 실전 패턴: 딕셔너리에 key가 있으면 그 값을, 없으면 기본값을 쓰기
config = {'model': 'gpt-4o-mini'}

try:
    temp = config['temperature']
except KeyError:
    temp = 0.5   # 기본값

print('temperature =', temp)

# 사실 이런 케이스는 .get()이 더 간결하다 — 상황에 맞게 선택
print('temperature(get) =', config.get('temperature', 0.5))

## 8. 파일 입출력 — 텍스트 & JSON 저장/로드

LLM 서비스에서 파일 I/O가 필요한 순간:
- 대화 이력을 저장해서 다음 세션에 불러오기
- FAQ 데이터셋(JSON)을 로드하기
- 모델 응답을 로그로 남기기

`with open(...) as f:` 구문은 자동으로 파일을 닫아준다 — 안전하다.
- `'w'` = 쓰기 (기존 내용 덮어씀)
- `'r'` = 읽기
- `'a'` = 이어쓰기

**비유 한 줄**: `with open`은 도서관에서 책을 빌릴 때 "반납까지 자동으로 챙겨주는 사서"가 옆에 있는 것과 같다.

In [ ]:
# 텍스트 파일 쓰기
with open('data.txt', 'w') as f:
    f.write('안녕하세요~!')

# 텍스트 파일 읽기
with open('data.txt', 'r') as f:
    content = f.read()

print('파일 내용:', content)

### JSON — 딕셔너리를 파일로 저장하는 표준
딕셔너리/리스트를 그대로 텍스트에 써버리면 다시 불러올 때 재조립이 번거롭다. JSON은 구조를 유지한 채 저장/복원하는 표준 포맷이다.

- `json.dump(data, f)` — 파일에 쓰기
- `json.load(f)` — 파일에서 읽기
- `ensure_ascii=False` — 한글이 `\uXXXX`로 깨지지 않게
- `indent=2` — 사람이 보기 좋게 들여쓰기

In [ ]:
import json
import os

# 폴더가 없으면 만들고, 있어도 에러 안 나게 exist_ok=True
os.makedirs('sample_output', exist_ok=True)

# 저장할 메시지 리스트 (어제 본 OpenAI messages 구조 그대로)
messages = [
    {'role': 'user', 'content': '비밀번호를 잊어버렸어요.'},
    {'role': 'assistant', 'content': '비밀번호 재설정 방법은...'}
]

# JSON으로 저장 — 한글이 그대로 보이도록 ensure_ascii=False
with open('sample_output/messages.json', 'w', encoding='utf-8') as f:
    json.dump(messages, f, ensure_ascii=False, indent=2)

# 다시 읽어서 복원 확인
with open('sample_output/messages.json', 'r', encoding='utf-8') as f:
    loaded = json.load(f)

print('복원된 데이터:')
for m in loaded:
    print(f"  [{m['role']}] {m['content']}")

## 9. 종합 연습 — 오늘 배운 걸로 미니 유틸 세트 만들기

오늘 배운 조각들(f-string / 리스트 / for / dict / 함수 / try-except / JSON)을 한꺼번에 써서, **간단한 대화 로거**를 만들어본다.

In [ ]:
import json

def make_message(role, content):
    """role/content를 받아 OpenAI 호환 메시지 딕셔너리를 만든다."""
    return {'role': role, 'content': content}

def save_messages(messages, path='sample_output/chat_log.json'):
    """메시지 리스트를 JSON 파일로 저장. 실패해도 프로그램은 죽지 않게 try."""
    try:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(messages, f, ensure_ascii=False, indent=2)
        return True
    except Exception as e:
        print('저장 실패:', e)
        return False

def load_messages(path='sample_output/chat_log.json'):
    """파일이 없거나 깨져도 빈 리스트를 돌려준다 — 서비스 무중단 핵심."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return []

# 사용 예시 — 오늘 만든 모든 조각이 여기 모인다
chat = [
    make_message('system',    '당신은 IT 상담원입니다.'),
    make_message('user',      '비밀번호를 잊어버렸어요.'),
    make_message('assistant', '비밀번호 재설정 방법을 안내드리겠습니다.'),
]

save_messages(chat)                       # 저장
restored = load_messages()                # 다시 로드

print(f'복원된 메시지 개수: {len(restored)}')
for i, msg in enumerate(restored, start=1):
    print(f'{i}. [{msg["role"]}] {msg["content"]}')

## 10. 오늘 정리

- **변수/타입**: `str/int/float/bool`, 필요하면 `type()`으로 확인.
- **f-string**: `f"{변수}"` — 프롬프트 조립의 핵심. LLM 서비스 = 문자열 조립력.
- **리스트**: `[]`, 0부터 인덱싱, `append()`로 추가, `enumerate()`로 인덱스까지.
- **딕셔너리**: `{key: value}`, `[]`는 없으면 에러 / `.get()`은 기본값 반환. **OpenAI 메시지 = 딕셔너리 of role/content**.
- **함수**: 반복 로직 포장. 기본값 매개변수로 유연하게.
- **try-except**: 네트워크·키 오류 등 예외 상황에서도 서비스가 멈추지 않게.
- **파일 I/O + JSON**: 대화 이력/데이터셋 저장 복원의 표준.

이제 내일(3/12)부터는 이 도구들로 OpenAI API를 본격 래핑하기 시작한다. messages 리스트를 동적으로 쌓고, 응답을 파싱해 파일로 남기는 '진짜 챗봇의 기본'을 구현한다.